# Train YOLOv8 dari anotasi feedback VoltRAB

Notebook ini mengikuti notebook referensi project: YOLOv8s, `imgsz=640`, validasi, dan export ONNX opset 12. Dataset dibentuk dari report user berstatus **reviewed/resolved** yang memiliki `system_report.training_annotations`. Hasil ONNX otomatis diunggah ke Supabase sebagai model **belum aktif**.

> Jangan tulis credential di cell. Tambahkan `SUPABASE_URL` dan `SUPABASE_SECRET_KEY` melalui panel **Colab Secrets**.

## 1. Aktifkan GPU
Pilih **Runtime → Change runtime type → T4 GPU**, lalu jalankan cell berikut.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics==8.3.191 onnx==1.17.0 onnxruntime==1.20.1 onnxslim==0.1.59 requests==2.32.3 Pillow==11.1.0 PyYAML==6.0.2

import ultralytics
ultralytics.checks()

## 2. Konfigurasi
Nilai ini aman untuk diubah. Untuk dataset kecil, jangan langsung mengaktifkan model hasil training sebelum diuji.

In [ ]:
from pathlib import Path

WORKDIR = Path('/content/voltrab-training')
BASE_MODEL = 'yolov8s.pt'
EPOCHS = 25
IMAGE_SIZE = 640
BATCH_SIZE = 8
MIN_REPORTS = 4
MODEL_NAME = 'VoltRAB feedback model'
MODEL_VERSION = 'feedback-v1'
WORKDIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from google.colab import userdata

SUPABASE_URL = userdata.get('SUPABASE_URL').rstrip('/')
SUPABASE_SECRET_KEY = userdata.get('SUPABASE_SECRET_KEY')
if not SUPABASE_URL or not SUPABASE_SECRET_KEY:
    raise RuntimeError('Tambahkan SUPABASE_URL dan SUPABASE_SECRET_KEY di Colab Secrets.')
print('Credential ditemukan tanpa menampilkan nilainya.')

## 3. Ambil report yang sudah direview admin
Report `open` sengaja tidak digunakan untuk mencegah data mentah langsung masuk training.

In [ ]:
import requests

HEADERS = {
    'apikey': SUPABASE_SECRET_KEY,
    'Authorization': f'Bearer {SUPABASE_SECRET_KEY}',
}
params = {
    'select': 'id,detection_image,system_report,created_at,status',
    'report_source': 'eq.user',
    'status': 'in.(reviewed,resolved)',
    'order': 'created_at.asc',
}
response = requests.get(
    f'{SUPABASE_URL}/rest/v1/detection_feedback_reports',
    headers=HEADERS, params=params, timeout=60,
)
response.raise_for_status()
reports = [
    report for report in response.json()
    if report.get('detection_image')
    and (report.get('system_report') or {}).get('training_annotations')
]
print(f'{len(reports)} report beranotasi siap diproses.')
if len(reports) < MIN_REPORTS:
    raise RuntimeError(f'Butuh minimal {MIN_REPORTS} report reviewed/resolved untuk train/validation.')

## 4. Bentuk dataset YOLO

In [ ]:
import base64
import io
import random
import shutil
import yaml
from PIL import Image as PILImage

DATASET = WORKDIR / 'dataset'
if DATASET.exists():
    shutil.rmtree(DATASET)
for split in ('train', 'val'):
    (DATASET / 'images' / split).mkdir(parents=True, exist_ok=True)
    (DATASET / 'labels' / split).mkdir(parents=True, exist_ok=True)

def read_image(value):
    if value.startswith('data:'):
        return base64.b64decode(value.split(',', 1)[1])
    result = requests.get(value, timeout=60)
    result.raise_for_status()
    return result.content

labels = sorted({
    str(annotation['label']).strip()
    for report in reports
    for annotation in report['system_report']['training_annotations']
    if str(annotation.get('label', '')).strip()
}, key=str.casefold)
class_ids = {name: index for index, name in enumerate(labels)}
if not labels:
    raise RuntimeError('Tidak ada label anotasi yang valid.')

random.Random(42).shuffle(reports)
val_count = max(1, round(len(reports) * 0.2))
for index, report in enumerate(reports):
    split = 'val' if index < val_count else 'train'
    image = PILImage.open(io.BytesIO(read_image(report['detection_image']))).convert('RGB')
    stem = str(report['id'])
    image.save(DATASET / 'images' / split / f'{stem}.jpg', quality=95)
    rows = []
    for annotation in report['system_report']['training_annotations']:
        label = str(annotation.get('label', '')).strip()
        bbox = annotation.get('bbox') or {}
        x, y = float(bbox.get('x', 0)), float(bbox.get('y', 0))
        width, height = float(bbox.get('width', 0)), float(bbox.get('height', 0))
        if label not in class_ids or width <= 0 or height <= 0:
            continue
        x_center = (x + width / 2) / 100
        y_center = (y + height / 2) / 100
        rows.append(f'{class_ids[label]} {x_center:.6f} {y_center:.6f} {width / 100:.6f} {height / 100:.6f}')
    if not rows:
        raise RuntimeError(f'Report {stem} tidak memiliki bounding box valid.')
    (DATASET / 'labels' / split / f'{stem}.txt').write_text('\n'.join(rows) + '\n')

dataset_yaml = {
    'path': str(DATASET),
    'train': 'images/train',
    'val': 'images/val',
    'names': {index: name for index, name in enumerate(labels)},
}
yaml_path = DATASET / 'data.yaml'
yaml_path.write_text(yaml.safe_dump(dataset_yaml, sort_keys=False, allow_unicode=True))
print(dataset_yaml)

## 5. Training dan validasi

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)
train_result = model.train(
    data=str(yaml_path), epochs=EPOCHS, imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE, plots=True, project=str(WORKDIR / 'runs'),
    name='feedback', seed=42,
)
best_pt = Path(train_result.save_dir) / 'weights' / 'best.pt'
if not best_pt.exists():
    raise RuntimeError('best.pt tidak ditemukan setelah training.')
validation = YOLO(str(best_pt)).val(data=str(yaml_path), imgsz=IMAGE_SIZE)

## 6. Export ONNX seperti notebook referensi

In [ ]:
trained_model = YOLO(str(best_pt))
onnx_path = Path(trained_model.export(
    format='onnx', opset=12, imgsz=[640, 640], simplify=True, dynamic=False,
))
print(f'ONNX selesai: {onnx_path.name} ({onnx_path.stat().st_size:,} bytes)')

## 7. Upload ke Supabase sebagai model draft
Model sengaja dibuat `is_active=false`. Uji dari dashboard admin sebelum mengaktifkannya.

In [ ]:
from datetime import datetime, timezone

timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
storage_path = f'training/{timestamp}-{onnx_path.name}'
upload_headers = {**HEADERS, 'Content-Type': 'application/octet-stream', 'x-upsert': 'false'}
with onnx_path.open('rb') as model_file:
    upload = requests.post(
        f'{SUPABASE_URL}/storage/v1/object/yolo-models/{storage_path}',
        headers=upload_headers, data=model_file, timeout=300,
    )
upload.raise_for_status()

metadata = {
    'name': MODEL_NAME,
    'description': f'Trained from {len(reports)} reviewed feedback reports; classes: {labels}',
    'version': MODEL_VERSION,
    'file_path': storage_path,
    'file_size': onnx_path.stat().st_size,
    'is_active': False,
}
insert = requests.post(
    f'{SUPABASE_URL}/rest/v1/yolo_models',
    headers={**HEADERS, 'Content-Type': 'application/json', 'Prefer': 'return=representation'},
    json=metadata, timeout=60,
)
if not insert.ok:
    requests.delete(
        f'{SUPABASE_URL}/storage/v1/object/yolo-models/{storage_path}',
        headers=HEADERS, timeout=60,
    )
    insert.raise_for_status()
created = insert.json()[0]
print(f"Model draft berhasil dibuat: {created['id']}")
print('Buka dashboard admin untuk menguji lalu mengaktifkan model.')